# Lesson 5

How well is your LLM doing? Does it meet an accuracy criteria?

1. Set up: create the chain you want to evaluate
2. Figure out what data points we want to evaluate on:
    
    Method 1. (most simple) - come up with data points that we think are good examples ourself - an example includes a question and an answer (creating validation data) - doesnt scale well
    
    Method 2. Automate it with lamguage models themself. We want to create a bunch of examples so we use the apply_and_parse method - get back a dictionary with query, answer pair
    Often when the wrong result is given, it's not necessarily the language model itself that's messing up, it's actually the retrieval step that's messing up - using the debug tool can help you see when this is happening.


## Benefits of using LLMs to evaluate over hard coded examples:
- You can see in the image below that the actual and predicted answers are very different, yet semantically similar, which makes manually checking accuracy very difficult (eg you can't use things like regex). Using a LLM to evaluate a model can help with this as it can check whether actual and preedicted answers are semantically similar.

![Alt text](../images/L5.1.png)

You can also use the Langchain evaluation platform which does everything that this notebook does, but in a UI, which is a lot more user friendly.

In [1]:
# Step 1: Environment Setup
# Load environment variables from .env file (contains API keys)
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

# Set the LLM model to use for evaluation
llm_model = "gpt-3.5-turbo"

## Create QandA application

In [2]:
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_openai import OpenAIEmbeddings

In [3]:
# Step 3: Load Documents
# Load the CSV file containing product information
# Each row becomes a Document object with page_content and metadata
file = '../.data/OutdoorClothingCatalog_L4.csv'
loader = CSVLoader(file_path=file)
data = loader.load()  # Returns a list of Document objects

In [4]:
# Step 4: Create Vector Store and Retriever
# This converts documents into embeddings and stores them for similarity search

# Initialize the embedding model (converts text to numerical vectors)
embeddings = OpenAIEmbeddings()

# Create vector store: embed all documents and store them in memory
# This enables semantic search - finding documents similar to a query
vectorstore = DocArrayInMemorySearch.from_documents(data, embeddings)

# Create a retriever: interface to search the vector store
# Will return the most relevant documents for a given query
retriever = vectorstore.as_retriever()

In [5]:
# Step 5: Create the Q&A Chain
# This is the application we want to evaluate - it answers questions using retrieved documents

# Initialize the LLM with temperature=0 for consistent, deterministic responses
llm = ChatOpenAI(temperature = 0.0, model=llm_model)

# Create RetrievalQA chain:
# - chain_type="stuff": Simplest method - stuffs all retrieved documents into the prompt
# - retriever: Used to find relevant documents for each question
# - verbose=True: Shows the chain's internal steps when running
# - document_separator: Separates multiple documents in the prompt
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"  # Separator between documents in prompt
    }
)

In [6]:
# Inspect a sample document to understand the data structure
# Each document has page_content (the text) and metadata (source, row number, etc.)
data[10]

Document(metadata={'source': '../.data/OutdoorClothingCatalog_L4.csv', 'row': 10}, page_content='name: Insulated Stainless Steel Bottle\ndescription: Keeps drinks cold for 24 hours and hot for 12, ideal for daily commutes.')

In [7]:
# View another sample document
data[11]

Document(metadata={'source': '../.data/OutdoorClothingCatalog_L4.csv', 'row': 11}, page_content='name: Smart Fitness Tracker\ndescription: Track your steps, heart rate, and sleep patterns with real-time sync.')

## Hard coded examples

In [8]:
# Step 6: Create Hard-Coded Evaluation Examples
# Method 1: Manually create question-answer pairs
# Each example has:
#   - "query": The question to ask
#   - "answer": The expected/correct answer
# This method doesn't scale well but is good for testing specific cases

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

## LLM-generated examples

In [9]:
# Step 7: Import Components for LLM-Generated Examples
# QAGenerateChain is not available in latest LangChain, so we'll build our own using LCEL
# ChatPromptTemplate: Create prompts with placeholders
# StrOutputParser: Extract text from LLM response objects
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [10]:
# Step 8: Create QA Generation Chain
# This chain will automatically generate question-answer pairs from documents
# Using LangChain Expression Language (LCEL) with the pipe operator (|)

# Define the prompt template that instructs the LLM to create Q&A pairs
qa_generation_prompt = ChatPromptTemplate.from_template(
    """You are an expert at creating test question-answer pairs.
Given a document, create a question and answer pair that tests understanding of the document.
Return your response in the following format:
QUESTION: [your question here]
ANSWER: [your answer here]

Document: {doc}
"""
)

# Chain components using pipeline operator (|):
# 1. qa_generation_prompt: Formats input dict into a prompt string
# 2. ChatOpenAI: Sends prompt to LLM, returns AIMessage object
# 3. StrOutputParser: Extracts text content from AIMessage
# The pipe operator chains these together: input → prompt → LLM → text output
example_gen_chain = qa_generation_prompt | ChatOpenAI(model=llm_model) | StrOutputParser()

In [11]:
# Step 9: Generate Examples Automatically
# Method 2: Use LLM to automatically create question-answer pairs from documents
# This scales better than manual creation

new_examples = []
# Process first 5 documents to generate test examples
for doc in data[:5]:
    # Invoke the chain: pass document content, get back formatted Q&A text
    result = example_gen_chain.invoke({"doc": doc.page_content})
    
    # Parse the result to extract question and answer
    # The LLM returns text in format: "QUESTION: ...\nANSWER: ..."
    lines = result.strip().split('\n')
    query = ""
    answer = ""
    for line in lines:
        if line.startswith("QUESTION:"):
            query = line.replace("QUESTION:", "").strip()
        elif line.startswith("ANSWER:"):
            answer = line.replace("ANSWER:", "").strip()
    
    # Only add if both question and answer were successfully extracted
    if query and answer:
        new_examples.append({"query": query, "answer": answer})

In [12]:
# View the first LLM-generated example
new_examples[0]

{'query': "What is the key feature of the Women's Campside Oxfords?",
 'answer': "The key feature of the Women's Campside Oxfords is their ultracomfortable lace-to-toe design that boasts a broken-in feel right out of the box."}

In [13]:
# Compare with the original document that was used to generate the example
data[0]

Document(metadata={'source': '../.data/OutdoorClothingCatalog_L4.csv', 'row': 0}, page_content="name: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a broken-in feel right out of the box.")

## Combine examples

In [14]:
# Step 10: Combine All Examples and Test the Q&A Chain
# Merge hard-coded and LLM-generated examples into one list
examples += new_examples

# Test the Q&A chain with the first example
# invoke() runs the chain and returns a dict with "query" and "result" keys
qa.invoke({"query": examples[0]["query"]})



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': "I don't have information about the Cozy Comfort Pullover Set to confirm if it has side pockets."}

## Manual evaluation

In [15]:
# Step 11: Enable Debug Mode for Manual Inspection
# Debug mode shows detailed information about what the chain is doing:
# - What documents were retrieved
# - What prompt was sent to the LLM
# - What the LLM returned
# This helps identify if retrieval is the problem vs. the LLM itself

import langchain

langchain.debug = True
qa.invoke({"query": examples[0]["query"]})



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': "I don't have information about the Cozy Comfort Pullover Set to confirm if it has side pockets."}

In [16]:
# Turn off debug mode to reduce output verbosity
langchain.debug = False

## LLM assisted evaluation

In [17]:
# Step 12: LLM-Assisted Evaluation
# Instead of manually checking if answers are correct, use an LLM to evaluate
# This is especially useful when answers are semantically similar but worded differently

# Part 1: Generate predictions for all examples
# Run the Q&A chain on each example to get predicted answers
predictions = []
for example in examples:
    result = qa.invoke({"query": example["query"]})
    # RetrievalQA.invoke returns a dict with "result" key containing the answer
    result_text = result.get("result", str(result)) if isinstance(result, dict) else str(result)
    predictions.append({
        "query": example["query"],
        "answer": example["answer"],  # The expected/correct answer
        "result": result_text  # The predicted answer from our Q&A chain
    })

# Part 2: Create an evaluation chain
# This chain uses an LLM to judge if predicted answers are correct
# QAEvalChain is not available in latest LangChain, so we build our own using LCEL
eval_prompt = ChatPromptTemplate.from_template(
    """You are an expert at evaluating the quality of question-answer pairs.
Given a question, the correct answer, and the predicted answer, determine if the predicted answer is correct.

Question: {query}
Correct Answer: {answer}
Predicted Answer: {result}

Respond with only "CORRECT" or "INCORRECT".
"""
)
# Chain: prompt → LLM → text parser
eval_chain = eval_prompt | ChatOpenAI(temperature=0, model=llm_model) | StrOutputParser()

# Part 3: Evaluate all predictions
# For each example, ask the evaluation LLM if the prediction is correct
graded_outputs = []
for i, example in enumerate(examples):
    grade = eval_chain.invoke({
        "query": example["query"],
        "answer": example["answer"],
        "result": predictions[i]["result"]
    })
    graded_outputs.append({"text": grade.strip()})  # Store the grade (CORRECT/INCORRECT)



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [18]:
# Step 13: Display Evaluation Results
# Print a summary showing:
# - The question asked
# - The expected/correct answer
# - The predicted answer from our Q&A chain
# - Whether the LLM evaluator marked it as CORRECT or INCORRECT
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: I don't know, would you like me to find out more information for you?
Predicted Grade: INCORRECT

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: I don't have information about the Ultra-Lofty 850 Stretch Down Hooded Jacket in my current context.
Predicted Grade: INCORRECT

Example 2:
Question: What is the key feature of the Women's Campside Oxfords?
Real Answer: The key feature of the Women's Campside Oxfords is their ultracomfortable lace-to-toe design that boasts a broken-in feel right out of the box.
Predicted Answer: The key feature of the Women's Campside Oxfords is their ultracomfortable broken-in feel right out of the box.
Predicted Grade: CORRECT

Example 3:
Question: What is the purpose of the Recycled Waterhog Dog Mat, Chevron Weave?
Real Answer: The purpos

In [19]:
# View the detailed evaluation result for the first example
# The "text" field contains "CORRECT" or "INCORRECT"
graded_outputs[0]

{'text': 'INCORRECT'}